# 06 — Full model (`model.py`)

`AlphaFold2FromScratch` is the orchestration boundary. It embeds a prepared feature batch, refines representations through recycling and the Evoformer, predicts residue geometry with the structure module, and exposes distance and confidence heads.

## Stage map

```text
A3M --> msa.py --> model-ready feature batch
                         |
                         v
                  InputEmbedder
                    m, z, e
                         |
             +-----------v------------------+
             | RecyclingEmbedder + Evoformer|
             |       repeat N times         |
             +-----------+------------------+
                         |
                 m[0] --> single s
                         |
                         v
                  StructureModule
                         |
            +------------+-------------+
            |            |             |
         frames T     C-alpha       refined s,z
            |                          |
            +--> structure       distogram + pLDDT
```

`model.py` is deliberately thin: it wires focused modules together and owns the recycle loop and output heads.

In [ ]:
import sys
import torch

sys.path.insert(0, "../src")
torch.manual_seed(0)

from af2_from_scratch import AF2Config
from af2_from_scratch.feature_extraction import msa_features, sample_batch
from af2_from_scratch import AlphaFold2FromScratch

## 1. Build a small model-ready batch

Feature extraction is data preparation outside `AlphaFold2FromScratch`. The model receives tensors rather than raw A3M text.

In [ ]:
cfg = AF2Config(
    c_m=32,
    c_z=32,
    c_e=16,
    c_s=64,
    heads=4,
    pair_heads=2,
    ipa_heads=2,
    c_hidden=8,
    n_evo=1,
    n_extra=1,
    n_ipa=1,
    n_clu=16,
    n_ext=16,
    recycles=1,
)
features = msa_features("../examples/tautomerase/alignment.a3m")
batch = sample_batch(features, cfg.n_clu, cfg.n_ext, mask_p=0.0, seed=0)
for name, value in batch.items():
    print(f"{name:16s} {tuple(value.shape)}")

## 2. Run the complete forward pass

The recycle loop repeatedly re-injects normalized previous representations before another Evoformer pass. Gradients do not flow through recycled inputs. The final query row `m[0]` becomes the single representation consumed by the structure module.

In [ ]:
model = AlphaFold2FromScratch(cfg).eval()
with torch.no_grad():
    output = model(batch)

print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
for name, value in output.items():
    print(f"{name:16s} {tuple(value.shape)}")

## 3. Read the outputs

```text
T              (R,4,4)    residue positions and orientations
ca             (R,3)      C-alpha coordinates = frame origins
disto_logits   (R,R,64)   predicted pairwise-distance bins
plddt_logits   (R,50)     predicted per-residue confidence bins
```

The architecture notebooks end here. Continue to notebook 07 for single-protein distillation and notebook 08 for held-out multi-protein evaluation.